# Breast Cancer

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from sklearn.metrics import adjusted_rand_score

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / "SpaDiff").is_dir()),
    cwd,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import SpaDiff as sd
from SpaDiff.spatial import spatial_reconstruction
from SpaDiff.utils import adjust_louvain_resolution, cal_purity, set_seed

## Configuration

In [ ]:
SEED = 42
SLICE_ORDER = ["H_1", "H_2", "H_3"]
REFERENCE_BATCH = "H_1"
TRUTH_KEY = "pathologist"
N_CLUSTERS = 7
K_INTRA = 5
K_INTER = 2
N_NEIGHBORS = 6
TOPOLOGY_MODE = "slice_aware"
TRAINING_EPOCHS = 500

MAX_ORDER = 2
SIMPLEX_ORDERS = (
    (0,) if MAX_ORDER == 0
    else tuple(range(1, MAX_ORDER + 1))
)
DSM_WEIGHTING = "variance"
PRIOR_KL_LOSS_WEIGHT = 0.0  

DATA_ROOT = Path("path/breastcancer/")
print("DATA_ROOT =", DATA_ROOT)

set_seed(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device =", device)

## Data loading

In [ ]:
samples = {}
for sample in SLICE_ORDER:
    current = sc.read_h5ad(DATA_ROOT / f"{sample}.h5ad")
    current.var_names_make_unique()

    sc.pp.normalize_total(current, target_sum=1e4)
    sc.pp.log1p(current)
    current, _ = spatial_reconstruction(current, alpha=1.0, n_neighbors=N_NEIGHBORS)

    samples[sample] = current

adata = sc.concat(samples, join="inner", label="batch_name", index_unique="-")
adata.obs["batch_name"] = pd.Categorical(adata.obs["batch_name"], categories=SLICE_ORDER, ordered=True)
adata.layers["counts"] = adata.X.copy()
print(adata)
print(adata.obs["batch_name"].value_counts())
if TRUTH_KEY not in adata.obs:
    print(f"Missing {TRUTH_KEY!r}; skipping purity and ARI.")

In [ ]:
sc.pp.highly_variable_genes(adata, flavor="seurat_v3", layer="counts",n_top_genes=3000, batch_key="batch_name")
adata = adata[:, adata.var["highly_variable"]].copy()

sc.tl.pca(adata, n_comps=50)
pca_array = np.ascontiguousarray(adata.obsm["X_pca"], dtype=np.float32)
features = torch.from_numpy(pca_array).to(device)

## Simplicial complex

In [ ]:
topology = sd.build_spatial_topology(
    adata,
    mode=TOPOLOGY_MODE,
    batch_key="batch_name",
    slice_order=SLICE_ORDER,
    k_intra=K_INTRA,
    k_inter=K_INTER,
    max_order=MAX_ORDER,
    device=device,
)
operators = topology.operators

## Conditional VP-SDE training

In [ ]:
config = sd.SpaDiffConfig(
    data_dim=features.shape[1],
    condition_input_dim=features.shape[1],
    score_depth=4,
    dropout=0.2,
    topology_projection_dropout=0.1,
    topology_residual=True,
    topology_output_normalization="feature",
    simplex_orders=SIMPLEX_ORDERS,
    propagation_steps=5,
    propagation_alpha=0.4,
    num_batches=len(SLICE_ORDER),
    dsm_weighting=DSM_WEIGHTING,
    prior_kl_weight=PRIOR_KL_LOSS_WEIGHT,
    batch_balanced_loss=True,
)
model = sd.SpaDiff(config).to(device)
adata = model.fit_transform(
    adata,
    features,
    operators,
    condition_features=features,
    batch_key="batch_name",
    batch_order=SLICE_ORDER,
    reference_batch=REFERENCE_BATCH,
    epochs=TRAINING_EPOCHS,
    progress=True,
    ode_steps=250,
)

In [ ]:
adata = sd.write_denoised_expression(
    adata,
    latent_key="X_spadiff",
    loadings_key="PCs",
    layer_key="spadiff_denoised",
    adjacency=topology.adjacency,
    smoothing_alpha=1.0,
    clip_nonnegative=True,
)
print("stored denoised layer:", "spadiff_denoised", adata.layers["spadiff_denoised"].shape)

## Louvain spatial domains

In [ ]:
sc.pp.neighbors(adata, use_rep='X_spadiff', n_neighbors=15, random_state=SEED)
sc.tl.louvain( adata, key_added="louvain", resolution=0.91, random_state=SEED)

In [ ]:
for sample in SLICE_ORDER:
    subset_obs = adata.obs.loc[adata.obs["batch_name"] == sample]

    valid = subset_obs[[TRUTH_KEY, "louvain"]].dropna()
    truth_codes = pd.Categorical(valid[TRUTH_KEY]).codes
    pred_codes = pd.Categorical(valid["louvain"]).codes
    purity = cal_purity(truth_codes, pred_codes)
    print(f"purity={purity:.3f}")

subset_obs = adata.obs.loc[adata.obs["batch_name"] == sample]
valid = subset_obs[[TRUTH_KEY, "louvain"]].dropna()
truth_codes = pd.Categorical(valid[TRUTH_KEY]).codes
pred_codes = pd.Categorical(valid["louvain"]).codes
purity = cal_purity(truth_codes, pred_codes)
print(f"all slices:purity={purity:.3f}")

In [ ]:
plot_color = ["#7495D3", "#59BE86", "#FEB915", "#C798EE", "#6D1A9C", "#F56867", "#D1D1D1"]
fig, axes = plt.subplots(1, len(SLICE_ORDER), figsize=(18, 5))
for axis, sample in zip(np.atleast_1d(axes), SLICE_ORDER):
    subset = adata[adata.obs["batch_name"] == sample].copy()
    sc.pl.spatial(
        subset, color="louvain", ax=axis, show=False, spot_size=280,
        palette=plot_color, title=sample,
        # legend_loc=None,
    )
plt.tight_layout()
plt.show()

In [ ]:
sc.tl.umap(adata, random_state=SEED)
colors = ["batch_name", "louvain"]
if TRUTH_KEY in adata.obs:
    colors.append(TRUTH_KEY)
sc.pl.umap(
    adata, color=colors, size=60,
    legend_fontsize=11, legend_fontoutline=2,
)

In [ ]:
genes = "CD24"
sc.pl.spatial(
    adata,
    color=genes,
    layer='spadiff_denoised',
    spot_size=280,                      
    color_map='inferno',                 
    legend_loc=None,
    frameon=False, 
)